In [ ]:
import pandas as pd
import os
from pathlib import Path 

ROOT_DIR = Path.cwd().parent.parent.parent

lkp_folder = os.path.join(ROOT_DIR, 'external_data','lookups','EHR','primary_care')

from pomegranate.db.ukbdb import UKBDatabase
Database = UKBDatabase()

from pomegranate.db.db_config import DB_TABLES

# Load maps and lookups into SQL

In [ ]:
def insertmany(db, table_name, df, chunk_size=10000, verbose=False):
    """ Insert many rows of data in the table.
    Returns: number of rows inserted. """

    # replace nan with None
    df = df.where(pd.notnull(df), None) 
    # convert df to list of lists
    data = [tuple(x) for x in df.to_numpy()]

    # Prepare the SQL query for inserting multiple rows
    # Assuming the table structure matches the data in `rows`
    placeholders = ','.join(['%s'] * len(data[0]))  # Create a placeholder for each column
    query = f"INSERT INTO {table_name} VALUES ({placeholders})"

    inserted_rows = 0
    with db.cursor as cur:
        for i in range(0, len(data), chunk_size):
            if i > 0 and verbose is True:
                print("-> ", end='', flush=True)
            chunk = data[i:i + chunk_size]
            # Execute the query and commit changes
            cur.executemany(query, chunk)
            db.commit()
            inserted_rows += cur.rowcount

    return inserted_rows

#### 1. dm+d lookup: `dmd_desc_type_lkp`
Comprehensive lookup of all dm+d releases' codes and descriptions, including a reference dm+d code of the most recent code per concept.

In [ ]:
# load data
dmd_key = pd.read_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), dtype=object)

Database = UKBDatabase()
# load table schema
Database.query(DB_TABLES['dmd_desc_type_lkp'])
# insert data
insertmany(Database, 'dmd_desc_type_lkp', dmd_key)

#### 2. dm+d to reference dm+d map: `dmd_reference_map`

A simple file mapping codes to their most recent code version - a simplification of `dmd_desc_type_lkp`

In [ ]:
dmd_ref_map = pd.read_csv(os.path.join(lkp_folder,'dmd_reference_code_map.csv.gz'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['dmd_reference_map'])
insertmany(Database, 'dmd_reference_map', dmd_ref_map)

#### 3. BNF lookup

In [ ]:
bnf_lkp = pd.read_csv(os.path.join(lkp_folder,'bnf_key.csv.gz'), compression='gzip', dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['bnf_desc_type_lkp'])
insertmany(Database, 'bnf_desc_type_lkp', bnf_lkp)

#### 4. BNF reformatting map: `bnf_formatted_map`

Converts BNF codes to the format in the BNF lookup (e.g., `01.03.01.00.00` → `010301`)

In [ ]:
bnf_formatted_map = pd.read_csv(os.path.join(lkp_folder,'bnf_from_entries_formatted_key.csv'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['bnf_formatted_map'])
insertmany(Database, 'bnf_formatted_map', bnf_formatted_map)
Database.query('DELETE FROM bnf_formatted_map WHERE original_bnf = formatted_bnf;') # QC

#### 5. BNF expansion map: `bnf_code_desc_lkp_map`

This updates entry codes with more granular codes based on the entry description and the original code

In [ ]:
bnf_code_desc_expansion_map = pd.read_csv(os.path.join(lkp_folder,'bnf_code_expansion_by_desc_map.csv'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['bnf_code_desc_expansion_map'])
insertmany(Database, 'bnf_code_desc_expansion_map', bnf_code_desc_expansion_map)
Database.query('DELETE FROM bnf_code_desc_expansion_map WHERE entry_bnf = lkp_bnf;') # QC

#### 6. BNF to dm+d map: `bnf_dmd_map`

In [ ]:
bnf_dmd_map = pd.read_csv(os.path.join(lkp_folder,'bnf_dmd_map_simple.csv'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['bnf_dmd_map'])
insertmany(Database, 'bnf_dmd_map', bnf_dmd_map)

#### 7. Welsh Read2 lookup: `welsh_read_desc_lkp`

In [ ]:
welsh_lkp = pd.read_csv(os.path.join(lkp_folder,'welsh_read_desc_lkp.csv.gz'), compression='gzip', dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['welsh_read_desc_lkp'])
insertmany(Database, 'welsh_read_desc_lkp', welsh_lkp)

#### 8. Welsh Read2 to dm+d map: `welsh_read_dmd_map`

In [ ]:
welsh_read_dmd_map = pd.read_csv(os.path.join(lkp_folder,'welsh_read_dmd_map_simple.csv'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['welsh_read_dmd_map'])
insertmany(Database, 'welsh_read_dmd_map', welsh_read_dmd_map)

### 9. dm+d AMP to VMP map: `dmd_amp_vmp_map`

In [ ]:
amp_vmp_map = pd.read_csv(os.path.join(lkp_folder,'dmd_amp_to_vmp_map.csv'), dtype=object)

Database = UKBDatabase()
Database.query(DB_TABLES['dmd_amp_vmp_map'])
insertmany(Database, 'dmd_amp_vmp_map', amp_vmp_map)